# Phase 4: Modeling
## Blood Cancer Detection — Hybrid Dataset

Models trained:
1. Logistic Regression
2. Decision Tree
3. Random Forest
4. SVM (RBF kernel)
5. MLP Neural Network (128 → 64 → 7)

**CV note:** 5-fold StratifiedKFold is run on the SMOTE-balanced training set (4,536 rows). The held-out test set (316 rows, original imbalanced distribution) is used only in Phase 5 evaluation — never seen during training or CV.

> **Synthetic-data caveat:** 12 of 15 numeric features are synthetic (see `docs/cbc_reference_ranges.md`). CV F1 scores reflect pipeline methodology, not clinical performance.

In [ ]:
import numpy as np
import pandas as pd
import joblib
import time
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.metrics import make_scorer, f1_score, classification_report

PROJECT = Path('..')
MODELS  = PROJECT / 'models'
REPORTS = PROJECT / 'reports'

X_train = np.load(MODELS / 'X_train.npy')
y_train = np.load(MODELS / 'y_train.npy')
X_test  = np.load(MODELS / 'X_test.npy')
y_test  = np.load(MODELS / 'y_test.npy')
le      = joblib.load(MODELS / 'label_encoder.pkl')

print(f'X_train : {X_train.shape}  y_train : {y_train.shape}')
print(f'X_test  : {X_test.shape}   y_test  : {y_test.shape}')
print(f'Classes : {list(le.classes_)}')


### Model Definitions

In [ ]:
MODELS_DEF = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, random_state=42, n_jobs=-1
    ),
    'Decision Tree': DecisionTreeClassifier(
        random_state=42
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, random_state=42, n_jobs=-1
    ),
    'SVM (RBF)': SVC(
        kernel='rbf', random_state=42, probability=True
    ),
    'MLP Neural Net': MLPClassifier(
        hidden_layer_sizes=(128, 64), max_iter=500,
        early_stopping=True, random_state=42
    ),
}

print('Models defined:')
for name in MODELS_DEF:
    print(f'  {name}')


### 5-Fold Stratified Cross-Validation

Scoring: accuracy, macro precision, macro recall, macro F1.  
All folds use the SMOTE-balanced training set only.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    'accuracy':  'accuracy',
    'precision': make_scorer(f1_score, average='macro', zero_division=0),
    'recall':    make_scorer(f1_score, average='macro', zero_division=0),
    'f1_macro':  make_scorer(f1_score, average='macro', zero_division=0),
}

results = {}
for name, clf in MODELS_DEF.items():
    t0 = time.time()
    cv_res = cross_validate(
        clf, X_train, y_train,
        cv=cv, scoring=scoring,
        return_train_score=False, n_jobs=-1
    )
    elapsed = time.time() - t0
    results[name] = {
        'accuracy_mean':  cv_res['test_accuracy'].mean(),
        'accuracy_std':   cv_res['test_accuracy'].std(),
        'f1_mean':        cv_res['test_f1_macro'].mean(),
        'f1_std':         cv_res['test_f1_macro'].std(),
        'time_s':         elapsed,
    }
    print(f'{name:<22}  F1={results[name]["f1_mean"]:.4f} '
          f'(+/-{results[name]["f1_std"]:.4f})  '
          f'Acc={results[name]["accuracy_mean"]:.4f}  '
          f'[{elapsed:.1f}s]')


### CV Leaderboard

In [ ]:
leaderboard = pd.DataFrame({
    name: {
        'CV Accuracy (mean)': f'{v["accuracy_mean"]:.4f}',
        'CV Accuracy (std)':  f'{v["accuracy_std"]:.4f}',
        'CV F1 Macro (mean)': f'{v["f1_mean"]:.4f}',
        'CV F1 Macro (std)':  f'{v["f1_std"]:.4f}',
        'CV Time (s)':        f'{v["time_s"]:.1f}',
    }
    for name, v in results.items()
}).T.sort_values('CV F1 Macro (mean)', ascending=False)

print(leaderboard.to_string())
best_name = leaderboard.index[0]
print(f'\nBest model by CV F1 Macro: {best_name}')


### ⚠️ Interpreting These CV Scores

> **Why are these scores so high (0.98–0.99)?**

The 12 augmented CBC columns (`Blasts_pct`, `Lymphocytes_pct`, `Hemoglobin_g_dL`, etc.) were **generated by sampling Gaussian distributions whose parameters were set per class**. This means the synthetic features encode the class label by construction — a model trained on them is partly learning the sampling rules, not real biological signal.

Concretely:
- `Blasts_pct` for ALL/AML was sampled from N(55, 20) — very different from Normal N(0.2, 0.2).
- `Lymphocytes_pct` for CLL/ALL was sampled from N(75, 8–10) vs. Normal N(30, 5).
- Any classifier can learn these rules with near-perfect accuracy.

**What these scores actually tell you:**
- The pipeline is correctly wired end-to-end.
- The preprocessing, SMOTE, and CV setup have no implementation bugs.
- Random Forest is the most stable model on this data (lowest std).

**What they do not tell you:**
- How well any model would perform on real unseen patients.
- Whether the feature set is clinically meaningful.

The per-class F1 breakdown below will confirm whether any individual class is 'unrealistically perfect' — a further indicator of synthetic over-separability.

### Per-Class F1 (via `cross_val_predict` — OOF predictions on training set)

In [ ]:
per_class_reports = {}
print('Computing out-of-fold predictions for per-class F1...')
for name, clf in MODELS_DEF.items():
    t0 = time.time()
    y_oof = cross_val_predict(clf, X_train, y_train, cv=cv)
    report = classification_report(
        y_train, y_oof,
        target_names=le.classes_,
        output_dict=True,
        zero_division=0,
    )
    per_class_reports[name] = report
    elapsed = time.time() - t0
    print(f'  {name} [{elapsed:.1f}s]')

print()
for name, report in per_class_reports.items():
    print(f'=== {name} ===')
    rows = {cls: report[cls] for cls in le.classes_}
    rows['macro avg'] = report['macro avg']
    df_report = pd.DataFrame(rows).T[['precision', 'recall', 'f1-score', 'support']]
    df_report = df_report.round(4)
    print(df_report.to_string())
    print()


### Refit Best Model on Full SMOTE Training Set

Final model is fit on all 4,536 training rows before evaluation on the test set.

In [ ]:
best_clf = MODELS_DEF[best_name]
best_clf.fit(X_train, y_train)

joblib.dump(best_clf, MODELS / 'best_model.pkl')
joblib.dump(best_name, MODELS / 'best_model_name.pkl')

print(f'Best model ({best_name}) fitted and saved -> models/best_model.pkl')


In [ ]:
all_fitted = {}
for name, clf in MODELS_DEF.items():
    if name == best_name:
        all_fitted[name] = best_clf
    else:
        clf.fit(X_train, y_train)
        all_fitted[name] = clf

joblib.dump(all_fitted, MODELS / 'all_models.pkl')
print('All fitted models saved -> models/all_models.pkl')


In [ ]:
report_path = REPORTS / 'phase4_modeling_summary.txt'
with open(report_path, 'w', encoding='utf-8') as f:
    f.write('=== PHASE 4: MODELING SUMMARY ===\n\n')
    f.write(f'Training set : {X_train.shape[0]} rows x {X_train.shape[1]} features (SMOTE-balanced)\n')
    f.write(f'Test set     : {X_test.shape[0]} rows x {X_test.shape[1]} features (original dist.)\n')
    f.write(f'CV strategy  : StratifiedKFold(n_splits=5, shuffle=True, random_state=42)\n\n')
    f.write('--- CV LEADERBOARD (sorted by F1 Macro) ---\n')
    f.write(leaderboard.to_string())
    f.write(f'\n\nBest model : {best_name}\n')
    f.write('Saved to   : models/best_model.pkl\n')
    f.write('\n\n--- PER-CLASS F1 (OOF cross_val_predict on SMOTE training set) ---\n')
    f.write('NOTE: High scores expected — synthetic CBC columns encode class by construction.\n\n')
    for name, report in per_class_reports.items():
        f.write(f'=== {name} ===\n')
        rows = {cls: report[cls] for cls in le.classes_}
        rows['macro avg'] = report['macro avg']
        df_r = pd.DataFrame(rows).T[['precision', 'recall', 'f1-score', 'support']].round(4)
        f.write(df_r.to_string())
        f.write('\n\n')

print(f'Report -> {report_path}')


### SMOTE Placement & Leakage Check

Loads saved artifacts from `models/` and verifies:
1. Test-set class counts match the pre-SMOTE stratified split exactly.
2. SMOTE was applied only to the training set (test row count unchanged).
3. No synthetic rows leaked into the test set.

In [ ]:
import numpy as np, joblib, pandas as pd
from pathlib import Path

_M  = Path('..') / 'models'
_le = joblib.load(_M / 'label_encoder.pkl')

_X_train = np.load(_M / 'X_train.npy')   # post-SMOTE
_y_train = np.load(_M / 'y_train.npy')
_X_test  = np.load(_M / 'X_test.npy')    # original hold-out
_y_test  = np.load(_M / 'y_test.npy')

_train_counts = pd.Series(_le.inverse_transform(_y_train)).value_counts().sort_index()
_test_counts  = pd.Series(_le.inverse_transform(_y_test)).value_counts().sort_index()

# Expected test counts from the stratified split (Phase 3)
_expected_test = {
    'ALL': 28, 'AML': 25, 'CLL': 28, 'CML': 24,
    'Lymphoma': 23, 'Multiple Myeloma': 26, 'Normal': 162,
}

_counts_match   = all(_test_counts.get(c, -1) == e for c, e in _expected_test.items())
_test_unchanged = _X_test.shape[0] == 316      # pre-SMOTE test size
_train_smoted   = _X_train.shape[0] == 4536   # 7 classes x 648

print('=== (1) TEST-SET CLASS COUNTS (used for final evaluation) ===')
print(_test_counts.to_string())
print(f'Total test rows : {_X_test.shape[0]}')

print()
print('=== (2) SMOTE PLACEMENT ===')
print(f'Train rows after SMOTE : {_X_train.shape[0]}  (expect 4536) -> {_train_smoted}')
print(f'Test rows unchanged    : {_X_test.shape[0]}   (expect  316) -> {_test_unchanged}')
print()
print('Pipeline order in 02_preprocessing.ipynb:')
print('  1  train_test_split(stratify=y)        -- 316 test rows locked in')
print('  2  preprocessor.fit_transform(X_train) -- fitted on train only')
print('  3  preprocessor.transform(X_test)      -- transform only, no refit')
print('  4  SMOTE.fit_resample(X_train_t, ...)  -- SMOTE sees ONLY train')
print('  5  np.save(X_test.npy, X_test_t)       -- saved before any resampling')

print()
print('=== (3) NO LEAKAGE CHECK ===')
print(f'Test counts match Phase 3 pre-SMOTE snapshot : {_counts_match}')
for cls, exp in _expected_test.items():
    got = _test_counts.get(cls, -1)
    print(f'  {cls:<20} expected={exp:>3}  got={got:>3}  OK={got == exp}')

# Write report
_rep = Path('..') / 'reports' / 'leakage_check.txt'
with open(_rep, 'w', encoding='utf-8') as _f:
    _f.write('=== SMOTE PLACEMENT & LEAKAGE CHECK ===\n\n')
    _f.write(f'X_train shape (post-SMOTE) : {_X_train.shape}\n')
    _f.write(f'X_test  shape (original)   : {_X_test.shape}\n\n')
    _f.write('TEST-SET CLASS COUNTS\n')
    _f.write(_test_counts.to_string())
    _f.write(f'\nTotal: {_X_test.shape[0]}\n\n')
    _f.write('SMOTE FIT ON TRAIN ONLY\n')
    _f.write(f'  Train rows == 4536 : {_train_smoted}\n')
    _f.write(f'  Test  rows ==  316 : {_test_unchanged}\n\n')
    _f.write('COUNTS MATCH PRE-SMOTE SNAPSHOT\n')
    _f.write(f'  All match : {_counts_match}\n')
    for cls, exp in _expected_test.items():
        got = _test_counts.get(cls, -1)
        _f.write(f'  {cls:<20} expected={exp:>3}  got={got:>3}  OK={got == exp}\n')

print(f'\nReport written -> {_rep}')


## Phase 4 Summary

| Setting | Value |
|---|---|
| Training rows | 4,536 (SMOTE-balanced) |
| CV folds | 5-fold StratifiedKFold |
| Primary metric | F1 Macro |
| Models compared | 5 |
| Best model | see leaderboard above |
| Saved artifacts | `models/best_model.pkl`, `models/all_models.pkl` |

Proceed to **Phase 5** for full evaluation on the held-out test set: confusion matrix, per-class metrics, and all required course visuals.